In [ ]:
# ============================================================================
# System Info & Environment Setup
# ============================================================================

import os
import multiprocessing

# Get system info
NUM_CPUS = multiprocessing.cpu_count()
print(f"Available CPUs: {NUM_CPUS}")

# Check RAM
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / (1024**3)
    print(f"Total RAM: {RAM_GB:.1f} GB")
except:
    RAM_GB = 167
    print(f"Assumed RAM: {RAM_GB} GB")

# Enable TF32 for faster computation on A100
import torch
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✅ TF32 enabled for faster A100 computation")

# Optimize CUDA memory allocation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# CRITICAL: Enable tokenizer parallelism - uses ALL CPUs for tokenization
os.environ["TOKENIZERS_PARALLELISM"] = "true"
print(f"✅ Tokenizer parallelism enabled ({NUM_CPUS} CPUs)")

print("\n✅ Environment configured for A100")

Available CPUs: 12
Total RAM: 167.1 GB
✅ TF32 enabled for faster A100 computation
✅ Tokenizer parallelism enabled (12 CPUs)

✅ Environment configured for A100


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [ ]:
# ============================================================================
# Install Dependencies
# ============================================================================

!pip install -q transformers>=4.46.0
!pip install -q sentence-transformers>=3.3.0
!pip install -q beir>=2.0.0
!pip install -q datasets>=3.0.0
!pip install -q accelerate>=1.2.0
!pip install -q huggingface-hub>=0.20.0
!pip install -q psutil

print("✅ All dependencies installed!")

✅ All dependencies installed!


In [ ]:
# ============================================================================
# Mount Google Drive
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [ ]:
# ============================================================================
# Configuration - A100 Optimized
# ============================================================================

import multiprocessing
import os

# ============================================================================
# MODEL PATH - Fine-tuned E5 from Epoch 1 Checkpoint
# ============================================================================
# Update this path to match your checkpoint location
MODEL_PATH = "/content/drive/MyDrive/NQ_full_INFO-KD-train-e5-klD-2026-01-28/models/e5-finetuned/checkpoint-5030"

# ============================================================================
# EVALUATION SETTINGS
# ============================================================================
MAX_SEQ_LENGTH = 512  # Enhanced: Using 512 tokens for evaluation

# ============================================================================
# A100-OPTIMIZED BATCH SIZES (80GB VRAM)
# ============================================================================
QUERY_BATCH_SIZE = 512
CORPUS_BATCH_SIZE = 4096  # A100 80GB can handle large batches with FP16

# Dataloader workers - use all CPUs for tokenization
NUM_WORKERS = multiprocessing.cpu_count()

# Output directory
OUTPUT_DIR = "/content/drive/MyDrive/KD-INFONCE-E5_Finetuned_Evaluation_2026-01-29_Epoch2"

print("="*80)
print("A100 OPTIMIZED CONFIGURATION - BEIR NQ")
print("="*80)
print(f"Model path: {MODEL_PATH}")
print(f"Max Sequence Length: {MAX_SEQ_LENGTH} tokens")
print(f"\n📊 Batch Sizes (optimized for A100 80GB):")
print(f"  Query batch size: {QUERY_BATCH_SIZE}")
print(f"  Corpus batch size: {CORPUS_BATCH_SIZE}")
print(f"  Tokenizer workers: {NUM_WORKERS} CPUs (parallel tokenization)")
print(f"\n📁 Output directory: {OUTPUT_DIR}")

# Verify model folder exists
if os.path.exists(MODEL_PATH):
    print(f"\n✅ Model folder found!")
    print(f"   Contents: {os.listdir(MODEL_PATH)}")
else:
    print(f"\n❌ Model folder NOT found at: {MODEL_PATH}")
    print("   Please update MODEL_PATH to your correct Google Drive path")

A100 OPTIMIZED CONFIGURATION - BEIR NQ
Model path: /content/drive/MyDrive/NQ_full_INFO-KD-train-e5-klD-2026-01-28/models/e5-finetuned/checkpoint-5030
Max Sequence Length: 512 tokens

📊 Batch Sizes (optimized for A100 80GB):
  Query batch size: 512
  Corpus batch size: 4096
  Tokenizer workers: 12 CPUs (parallel tokenization)

📁 Output directory: /content/drive/MyDrive/KD-INFONCE-E5_Finetuned_Evaluation_2026-01-29_Epoch2

✅ Model folder found!
   Contents: ['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', 'modules.json', 'README.md', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']


In [ ]:
# ============================================================================
# Imports & Device Setup
# ============================================================================

import json
import os
from pathlib import Path
from typing import Dict
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__} | Device: {device}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")

    if 'A100' in gpu_name:
        print("\n✅ A100 detected - using optimized settings")
    else:
        print(f"\n⚠️ Expected A100 but got {gpu_name}")

PyTorch: 2.9.0+cu126 | Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB

✅ A100 detected - using optimized settings


In [ ]:
# ============================================================================
# Load Model (directly from checkpoint folder)
# ============================================================================
# Matches the reference notebook exactly
# ============================================================================

print("Loading model...")
print(f"  Path: {MODEL_PATH}")
print(f"  Files: {os.listdir(MODEL_PATH)}")

# Load directly from the checkpoint folder (no assembly needed)
model = SentenceTransformer(MODEL_PATH)
model = model.to(device)
model = model.half()  # FP16 for 2x faster encoding

# Set max sequence length for evaluation (512 tokens)
model.max_seq_length = MAX_SEQ_LENGTH

print(f"\n✅ Model loaded")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Precision: FP16 (half)")
print(f"   Embedding dim: {model.get_sentence_embedding_dimension()}")

Loading model...
  Path: /content/drive/MyDrive/NQ_full_INFO-KD-train-e5-klD-2026-01-28/models/e5-finetuned/checkpoint-5030
  Files: ['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', 'modules.json', 'README.md', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']

✅ Model loaded
   Parameters: 109,482,240
   Precision: FP16 (half)
   Embedding dim: 768


In [ ]:
# ============================================================================
# Download and Load BEIR NQ Dataset
# ============================================================================

print("Downloading BEIR NQ dataset...")
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip"
data_path = util.download_and_unzip(url, "datasets")

print("Loading dataset...")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"\n✅ Dataset loaded:")
print(f"  Corpus: {len(corpus):,} documents")
print(f"  Queries: {len(queries):,}")
print(f"  Qrels: {len(qrels):,}")

corpus_size = len(corpus)
embedding_dim = model.get_sentence_embedding_dimension()
estimated_memory_gb = (corpus_size * embedding_dim * 2) / (1024**3)
print(f"\n📊 Estimated corpus embeddings size: {estimated_memory_gb:.2f} GB (FP16)")

datasets/nq.zip:   0%|          | 0.00/475M [00:00<?, ?iB/s]

Loading dataset...


  0%|          | 0/2681468 [00:00<?, ?it/s]


✅ Dataset loaded:
  Corpus: 2,681,468 documents
  Queries: 3,452
  Qrels: 3,452

📊 Estimated corpus embeddings size: 3.84 GB (FP16)


In [ ]:
# ============================================================================
# Verification: Sample Queries
# ============================================================================

import random

print("\n" + "="*80)
print("SAMPLE QUERIES WITH RELEVANT DOCUMENTS")
print("="*80)

sample_qids = random.sample(list(qrels.keys()), min(3, len(qrels)))

for i, qid in enumerate(sample_qids, 1):
    print(f"\n{'─'*80}")
    print(f"Example {i}:")
    print(f"{'─'*80}")
    print(f"\n📌 Query ID: {qid}")
    print(f"   Query Text: {queries[qid]}")

    print(f"\n📄 Relevant Document(s):")
    for doc_id, score in qrels[qid].items():
        if doc_id in corpus:
            doc = corpus[doc_id]
            text = doc.get('text', '')[:300]
            print(f"   Doc ID: {doc_id} | Text: {text}...")

print(f"\n✅ Verification complete")


SAMPLE QUERIES WITH RELEVANT DOCUMENTS

────────────────────────────────────────────────────────────────────────────────
Example 1:
────────────────────────────────────────────────────────────────────────────────

📌 Query ID: test1182
   Query Text: when are the next commonwealth games going to be held

📄 Relevant Document(s):
   Doc ID: doc41944 | Text: The Games are expected to take place between 27 July and 7 August 2022. The city was announced as the host at a press conference at the Arena Academy in Birmingham on 21 December 2017.[2]...

────────────────────────────────────────────────────────────────────────────────
Example 2:
────────────────────────────────────────────────────────────────────────────────

📌 Query ID: test1253
   Query Text: where is gulf breeze fl on the map

📄 Relevant Document(s):
   Doc ID: doc44275 | Text: Gulf Breeze is a city on the Fairpoint Peninsula in Santa Rosa County, Florida, United States and is a suburb of Pensacola which lies to the north, acro

In [ ]:
# ============================================================================
# Prepare Texts with E5 Prefixes
# ============================================================================

print("Preparing texts...")

query_ids = list(queries.keys())
query_texts = ["query: " + queries[qid] for qid in query_ids]
print(f"  Queries prepared: {len(query_texts):,}")

corpus_ids = list(corpus.keys())
corpus_texts = ["passage: " + corpus[cid]["title"] + " " + corpus[cid]["text"] for cid in corpus_ids]
print(f"  Corpus prepared: {len(corpus_texts):,}")

print("\n✅ Texts prepared with E5 prefixes")

Preparing texts...
  Queries prepared: 3,452
  Corpus prepared: 2,681,468

✅ Texts prepared with E5 prefixes


In [ ]:
# ============================================================================
# Encode Queries
# ============================================================================

print("\n" + "="*80)
print("ENCODING QUERIES")
print("="*80)

print(f"Encoding {len(query_texts):,} queries (batch_size={QUERY_BATCH_SIZE})...")

start_time = time.time()
query_embeddings = model.encode(
    query_texts,
    batch_size=QUERY_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_tensor=False,
    normalize_embeddings=True,
)
query_time = time.time() - start_time

query_embeddings = torch.from_numpy(query_embeddings).half()
print(f"\n✅ Query embeddings: {query_embeddings.shape}")
print(f"   Time: {query_time:.1f}s ({len(query_texts)/query_time:.0f} queries/sec)")


ENCODING QUERIES
Encoding 3,452 queries (batch_size=512)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Query embeddings: torch.Size([3452, 768])
   Time: 1.3s (2744 queries/sec)


In [ ]:
# ============================================================================
# Encode Corpus - Chunked with Visible Progress
# ============================================================================

print("\n" + "="*80)
print("ENCODING CORPUS - CHUNKED")
print("="*80)

print(f"\n🔧 Encoding settings:")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Batch size: {CORPUS_BATCH_SIZE}")
print(f"   Precision: FP16")

# Process in chunks to show progress and avoid memory issues
CHUNK_SIZE = 800_000  # 500K docs per chunk
num_chunks = (len(corpus_texts) + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"\nEncoding {len(corpus_texts):,} documents in {num_chunks} chunks...")
print(f"Expected time: ~5-8 minutes on A100\n")

torch.cuda.empty_cache()
start_time = time.time()

all_embeddings = []

for chunk_idx in range(num_chunks):
    chunk_start = chunk_idx * CHUNK_SIZE
    chunk_end = min(chunk_start + CHUNK_SIZE, len(corpus_texts))
    chunk_texts = corpus_texts[chunk_start:chunk_end]

    print(f"📦 Chunk {chunk_idx + 1}/{num_chunks}: Encoding {len(chunk_texts):,} documents...")
    chunk_start_time = time.time()

    chunk_embeddings = model.encode(
        chunk_texts,
        batch_size=CORPUS_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True,
    )
    all_embeddings.append(chunk_embeddings)

    chunk_time = time.time() - chunk_start_time
    elapsed = time.time() - start_time
    docs_done = chunk_end
    docs_per_sec = docs_done / elapsed
    remaining = (len(corpus_texts) - docs_done) / docs_per_sec if docs_per_sec > 0 else 0
    print(f"   ✓ Chunk finished in {chunk_time:.1f}s")
    print(f"   ✓ Progress: {docs_done:,}/{len(corpus_texts):,} | rate: {docs_per_sec:.0f} docs/s | ETA: {remaining/60:.1f} min\n")

corpus_time = time.time() - start_time
corpus_embeddings = torch.from_numpy(np.vstack(all_embeddings)).half()

print(f"\n✅ Corpus embeddings: {corpus_embeddings.shape}")
print(f"   Time: {corpus_time/60:.1f} minutes ({len(corpus_texts)/corpus_time:.0f} docs/sec)")
print(f"   Memory: {corpus_embeddings.element_size() * corpus_embeddings.numel() / 1e9:.2f} GB")

# ============================================================================
# IMMEDIATE SAVE: Checkpoint Corpus Embeddings
# ============================================================================
print("\n" + "="*80)
print("SAVING INTERMEDIATE CHECKPOINT")
print("="*80)

checkpoint_dir = Path(OUTPUT_DIR) / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True, parents=True)
checkpoint_path = checkpoint_dir / 'corpus_embeddings_nq_checkpoint.npy'

print(f"Saving checkpoint to: {checkpoint_path}...")
np.save(checkpoint_path, corpus_embeddings.cpu().numpy())
print("✅ Checkpoint saved! You can restart from here if needed.")


ENCODING CORPUS - CHUNKED

🔧 Encoding settings:
   GPU: NVIDIA A100-SXM4-80GB
   Batch size: 4096
   Precision: FP16

Encoding 2,681,468 documents in 4 chunks...
Expected time: ~5-8 minutes on A100

📦 Chunk 1/4: Encoding 800,000 documents...


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

   ✓ Chunk finished in 531.5s
   ✓ Progress: 800,000/2,681,468 | rate: 1505 docs/s | ETA: 20.8 min

📦 Chunk 2/4: Encoding 800,000 documents...


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

   ✓ Chunk finished in 495.9s
   ✓ Progress: 1,600,000/2,681,468 | rate: 1557 docs/s | ETA: 11.6 min

📦 Chunk 3/4: Encoding 800,000 documents...


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

   ✓ Chunk finished in 501.2s
   ✓ Progress: 2,400,000/2,681,468 | rate: 1570 docs/s | ETA: 3.0 min

📦 Chunk 4/4: Encoding 281,468 documents...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

   ✓ Chunk finished in 171.2s
   ✓ Progress: 2,681,468/2,681,468 | rate: 1577 docs/s | ETA: 0.0 min


✅ Corpus embeddings: torch.Size([2681468, 768])
   Time: 28.3 minutes (1577 docs/sec)
   Memory: 4.12 GB

SAVING INTERMEDIATE CHECKPOINT
Saving checkpoint to: /content/drive/MyDrive/KD-INFONCE-E5_Finetuned_Evaluation_2026-01-29_Epoch2/checkpoints/corpus_embeddings_nq_checkpoint.npy...
✅ Checkpoint saved! You can restart from here if needed.


In [ ]:
# ============================================================================
# Compute Evaluation Metrics - GPU Accelerated
# ============================================================================

print("\n" + "="*80)
print("COMPUTING METRICS - GPU ACCELERATED")
print("="*80)

print("Moving embeddings to GPU...")
query_embeddings_gpu = query_embeddings.to(device)
corpus_embeddings_gpu = corpus_embeddings.to(device)

print(f"GPU Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

ndcg_scores = {k: [] for k in [1, 5, 10]}
recall_scores = {k: [] for k in [1, 5, 10]}
mrr_scores = []

print(f"\nEvaluating {len(query_ids):,} queries...")
start_time = time.time()

EVAL_BATCH_SIZE = 100

for batch_start in tqdm(range(0, len(query_ids), EVAL_BATCH_SIZE)):
    batch_end = min(batch_start + EVAL_BATCH_SIZE, len(query_ids))

    batch_query_emb = query_embeddings_gpu[batch_start:batch_end]
    similarities = (batch_query_emb @ corpus_embeddings_gpu.T)
    top_indices = torch.topk(similarities, k=10, dim=1).indices.cpu().tolist()

    for i, idx in enumerate(range(batch_start, batch_end)):
        qid = query_ids[idx]

        if qid not in qrels or len(qrels[qid]) == 0:
            continue

        top_corpus_ids = [corpus_ids[j] for j in top_indices[i]]
        relevant_docs = set(qrels[qid].keys())

        for rank, cid in enumerate(top_corpus_ids, 1):
            if cid in relevant_docs:
                mrr_scores.append(1.0 / rank)
                break
        else:
            mrr_scores.append(0.0)

        for k in [1, 5, 10]:
            retrieved = top_corpus_ids[:k]
            recall = len(set(retrieved) & relevant_docs) / len(relevant_docs) if relevant_docs else 0
            recall_scores[k].append(recall)

            relevance_scores = [qrels[qid].get(cid, 0) for cid in retrieved]
            ideal_scores = sorted([qrels[qid].get(cid, 0) for cid in relevant_docs], reverse=True)[:k]

            if sum(ideal_scores) > 0:
                dcg = sum([rel / np.log2(rank + 2) for rank, rel in enumerate(relevance_scores)])
                idcg = sum([rel / np.log2(rank + 2) for rank, rel in enumerate(ideal_scores)])
                ndcg_scores[k].append(dcg / idcg if idcg > 0 else 0)

eval_time = time.time() - start_time
print(f"\n⏱️ Evaluation time: {eval_time:.1f}s")

metrics = {}
for k in [1, 5, 10]:
    metrics[f'nDCG@{k}'] = np.mean(ndcg_scores[k]) if ndcg_scores[k] else 0
    metrics[f'Recall@{k}'] = np.mean(recall_scores[k]) if recall_scores[k] else 0
metrics['MRR@10'] = np.mean(mrr_scores) if mrr_scores else 0

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
for metric in ['nDCG@1', 'nDCG@5', 'nDCG@10', 'Recall@1', 'Recall@5', 'Recall@10', 'MRR@10']:
    print(f"  {metric:<15}: {metrics[metric]:.4f}")


COMPUTING METRICS - GPU ACCELERATED
Moving embeddings to GPU...
GPU Memory used: 4.35 GB

Evaluating 3,452 queries...


  0%|          | 0/35 [00:00<?, ?it/s]


⏱️ Evaluation time: 0.6s

FINAL RESULTS
  nDCG@1         : 0.3914
  nDCG@5         : 0.5367
  nDCG@10        : 0.5749
  Recall@1       : 0.3492
  Recall@5       : 0.6691
  Recall@10      : 0.7795
  MRR@10         : 0.5256


In [ ]:
# ============================================================================
# Comparison with Baseline
# ============================================================================

print("\n" + "="*80)
print("COMPARISON WITH BASELINE (E5-base vanilla)")
print("="*80)
baseline = {
    'nDCG@10': 0.5997,
    'Recall@10': 0.8031,
    'MRR@10': 0.5493
}
print(f"{'Metric':<15} {'Baseline':<12} {'Finetuned':<12} {'Delta':<12} {'Change'}")
print("-"*60)
for metric in ['nDCG@10', 'Recall@10', 'MRR@10']:
    delta = metrics[metric] - baseline[metric]
    pct = (delta / baseline[metric]) * 100
    arrow = "↑" if delta > 0 else "↓" if delta < 0 else "→"
    print(f"{metric:<15} {baseline[metric]:<12.4f} {metrics[metric]:<12.4f} {delta:+.4f}      {arrow} {abs(pct):.1f}%")


COMPARISON WITH BASELINE (E5-base vanilla)
Metric          Baseline     Finetuned    Delta        Change
------------------------------------------------------------
nDCG@10         0.5997       0.5749       -0.0248      ↓ 4.1%
Recall@10       0.8031       0.7795       -0.0236      ↓ 2.9%
MRR@10          0.5493       0.5256       -0.0237      ↓ 4.3%


In [ ]:
# ============================================================================
# Save Results and Embeddings
# ============================================================================

print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

output_path = Path(OUTPUT_DIR)
output_path.mkdir(exist_ok=True, parents=True)

results = {
    'metrics': metrics,
    'model_path': MODEL_PATH,
    'num_queries': len(query_ids),
    'num_corpus': len(corpus_ids),
    'evaluation_type': 'BEIR NQ (test split)',
    'query_batch_size': QUERY_BATCH_SIZE,
    'corpus_batch_size': CORPUS_BATCH_SIZE,
    'encoding_time_corpus_minutes': corpus_time / 60,
    'encoding_time_query_seconds': query_time,
    'evaluation_time_seconds': eval_time,
    'precision': 'FP16',
    'environment': 'Colab A100',
    'parallel_tokenization': True,
    'num_cpus': multiprocessing.cpu_count()
}

with open(output_path / 'evaluation_results_nq.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"✅ Results saved: {output_path / 'evaluation_results_nq.json'}")

embeddings_dir = output_path / 'corpus_embeddings_nq'
embeddings_dir.mkdir(exist_ok=True, parents=True)

np.save(embeddings_dir / 'corpus_embeddings_fp16.npy', corpus_embeddings.cpu().numpy())
print(f"✅ Corpus embeddings saved: {embeddings_dir / 'corpus_embeddings_fp16.npy'}")

with open(embeddings_dir / 'corpus_ids.json', 'w') as f:
    json.dump(corpus_ids, f)
print(f"✅ Corpus IDs saved")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE!")
print("="*80)
print(f"\n📊 Summary:")
print(f"   Corpus: {len(corpus_ids):,} documents")
print(f"   Queries: {len(query_ids):,}")
print(f"   Total time: {(corpus_time + query_time + eval_time)/60:.1f} minutes")


SAVING RESULTS
✅ Results saved: /content/drive/MyDrive/KD-INFONCE-E5_Finetuned_Evaluation_2026-01-29_Epoch2/evaluation_results_nq.json
✅ Corpus embeddings saved: /content/drive/MyDrive/KD-INFONCE-E5_Finetuned_Evaluation_2026-01-29_Epoch2/corpus_embeddings_nq/corpus_embeddings_fp16.npy
✅ Corpus IDs saved

✅ EVALUATION COMPLETE!

📊 Summary:
   Corpus: 2,681,468 documents
   Queries: 3,452
   Total time: 28.4 minutes


In [ ]:
from google.colab import runtime
# Terminate the session to release GPU
runtime.unassign()